# Advanced Retrieval Pipeline with Metadata Filtering and Query Refinement

This notebook implements a retrieval system that:
1.  Analyzes user queries to extract metadata filters (Year, Dept, etc.).
2.  Refines the query for better semantic matching.
3.  Performs a filtered similarity search in ChromaDB.

In [1]:
import os
import json
from typing import List, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from pydantic import BaseModel, Field

# Load environment variables
load_dotenv("../.env")

# Configuration
DB_DIR = "../metadata_n_db/chroma_db"

if not os.path.exists(DB_DIR):
    print(f"Warning: DB directory {DB_DIR} does not exist. Please check the path.")
else:
    print(f"DB Directory: {os.path.abspath(DB_DIR)}")

DB Directory: /home/rishabh/coding/pro/RAG/metadata_n_db/chroma_db


In [2]:
class RAGRetriever:
    def __init__(self, db_dir: str, api_key_env: str = "GEMINI1", model_name: str = "gemini-1.5-flash"):
        """
        Initialize the RAG Retriever with Embeddings, Vector Store, BM25, and LLM.
        """
        self.api_key = os.getenv(api_key_env)
        if not self.api_key:
            raise ValueError(f"API Key environment variable '{api_key_env}' not found.")
            
        print(f"Initializing RAGRetriever with model: {model_name}")
        
        # 1. Initialize LLM
        self.llm = ChatGoogleGenerativeAI(
            model=model_name,
            temperature=0,
            google_api_key=self.api_key
        )
        
        # 2. Initialize Embeddings
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            google_api_key=self.api_key
        )
        
        # 3. Load Vector Store
        self.vectorstore = Chroma(
            persist_directory=db_dir,
            embedding_function=self.embeddings
        )
        print(f"Vector Store loaded with {self.vectorstore._collection.count()} documents.")
        
        # 4. Initialize BM25 (Keyword Search)
        print("Building BM25 Index from Vector Store documents...")
        self._build_bm25_index()
        
        # 5. Setup Chains
        self._setup_reranking_chain()
        
    def _build_bm25_index(self):
        """
        Fetches all documents from ChromaDB to build an in-memory BM25 index.
        Note: This approach works for small-to-medium datasets. For huge datasets, 
        consider a dedicated search engine like Elasticsearch or Solr.
        """
        try:
            # Fetch all docs from Chroma
            data = self.vectorstore.get() 
            docs = []
            # Reconstruct Document objects
            if data and data['documents']:
                for i, text in enumerate(data['documents']):
                    metadata = data['metadatas'][i] if data['metadatas'] else {}
                    docs.append(Document(page_content=text, metadata=metadata))
            
            if not docs:
                print("Warning: No documents found in Vector Store to build BM25 index.")
                self.bm25_retriever = None
                return

            self.bm25_retriever = BM25Retriever.from_documents(docs)
            print(f"BM25 Index built with {len(docs)} documents.")
            
        except Exception as e:
            print(f"Error building BM25 index: {e}")
            self.bm25_retriever = None

    def _setup_reranking_chain(self):
        """Sets up the LLM chain used for reranking documents."""
        
        # Output Schema
        class RelevanceScore(BaseModel):
            index: int = Field(description="The index of the document in the provided list")
            relevance_score: float = Field(description="A score from 0.0 to 1.0 indicating relevance")
            reasoning: str = Field(description="Brief reason why this document matches the constraints")

        class RankedDocuments(BaseModel):
            ranked_results: List[RelevanceScore]

        self.rerank_parser = JsonOutputParser(pydantic_object=RankedDocuments)

        # Prompt
        self.rerank_prompt = PromptTemplate(
            template="""You are an expert relevance ranker. 
            The user asked: "{query}"
            
            Below is a list of document snippets retrieved from a database. 
            Your job is to evaluate each snippet and determine if it truly answers the user's specific constraints (e.g., specific year, specific department, specific format).
            
            If a document is relevant, assign a high score (0.7 - 1.0).
            If it is topic-adjacent but misses the specific constraint (e.g., wrong year), assign a low score (0.0 - 0.3).
            
            Documents:
            {doc_list}
            
            Return the output as valid JSON matching the format instructions.
            {format_instructions}
            """,
            input_variables=["query", "doc_list"],
            partial_variables={"format_instructions": self.rerank_parser.get_format_instructions()},
        )

        self.rerank_chain = self.rerank_prompt | self.llm | self.rerank_parser

    def reciprocal_rank_fusion(self, results: List[List[Document]], k=60):
        """
        Combines multiple lists of ranked documents using Reciprocal Rank Fusion (RRF).
        Score = 1 / (rank + k)
        """
        fused_scores = {}
        doc_map = {} 

        for rank_list in results:
            for rank, doc in enumerate(rank_list):
                # Use page_content as a unique key (assuming no exact duplicates)
                # Ideally, use a unique ID if available in metadata
                doc_key = doc.page_content
                
                if doc_key not in fused_scores:
                    fused_scores[doc_key] = 0
                    doc_map[doc_key] = doc
                
                fused_scores[doc_key] += 1 / (rank + k)

        # Sort by fused score descending
        reranked_results = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        
        # Return list of Document objects
        return [doc_map[doc_str] for doc_str, score in reranked_results]

    def retrieve(self, query: str, k_fetch: int = 10, top_n: int = 3) -> List:
        """
        Retrieves documents using Hybrid Search (Vector + BM25) -> RRF -> LLM Reranking.
        """
        print(f"--- 1. Hybrid Retrieval for: '{query}' ---")
        
        # 1. Vector Search
        vector_docs = self.vectorstore.similarity_search(query, k=k_fetch)
        print(f"  [Vector] Found {len(vector_docs)} docs")
        
        # 2. Keyword Search (BM25)
        keyword_docs = []
        if self.bm25_retriever:
            self.bm25_retriever.k = k_fetch
            keyword_docs = self.bm25_retriever.invoke(query)
            print(f"  [Keyword] Found {len(keyword_docs)} docs")
        else:
            print("  [Keyword] BM25 not initialized, skipping.")
        
        # 3. Fusion (RRF)
        if keyword_docs:
            initial_docs = self.reciprocal_rank_fusion([vector_docs, keyword_docs], k=60)
            print(f"  [RRF] Fused into {len(initial_docs)} unique docs")
        else:
            initial_docs = vector_docs
            
        # Slice to k_fetch to keep the context window reasonable for reranker
        # (We might want slightly more than k_fetch if we fused, but let's keep it constrained)
        initial_docs = initial_docs[:k_fetch]
        
        print(f"\n[Log] Top {len(initial_docs)} Documents after Fusion:")
        for i, doc in enumerate(initial_docs):
            print(f"  [{i}] Source: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')} | Title: {doc.metadata.get('title')}")
        
        # 4. Format for the LLM
        doc_texts = []
        for i, doc in enumerate(initial_docs):
            snippet = f"Doc ID {i}:\nMetadata: {doc.metadata}\nContent: {doc.page_content[:400]}..." 
            doc_texts.append(snippet)
        
        combined_text = "\n\n".join(doc_texts)

        print(f"\n--- 2. Reranking {len(initial_docs)} documents ---")
        
        try:
            # 5. Call the LLM Judge
            ranking_result = self.rerank_chain.invoke({"query": query, "doc_list": combined_text})
            
            # 6. Sort and Filter
            sorted_ranks = sorted(ranking_result['ranked_results'], key=lambda x: x['relevance_score'], reverse=True)
            
            final_docs = []
            print("\n--- Top Selected Documents ---")
            for item in sorted_ranks[:top_n]:
                if item['relevance_score'] < 0.5:
                    print(f"Skipping Doc {item['index']} (Low Score: {item['relevance_score']})")
                    continue
                    
                # Map back to the document in initial_docs
                # Note: The LLM sees indices 0..N based on the list we sent it
                if 0 <= item['index'] < len(initial_docs):
                    original_doc = initial_docs[item['index']]
                    print(f"Score: {item['relevance_score']} | Doc Source: {original_doc.metadata.get('source')}")
                    print(f"Reasoning: {item['reasoning']}")
                    final_docs.append(original_doc)
                else:
                    print(f"Warning: LLM returned invalid index {item['index']}")
                
            return final_docs

        except Exception as e:
            print(f"Reranking failed: {e}. Falling back to raw search results.")
            return initial_docs[:top_n]

In [4]:
# Initialize the Retriever
# You can switch models here (e.g., "gemini-2.5-flash-lite" if available)
retriever = RAGRetriever(
    db_dir=DB_DIR, 
    api_key_env="GEMINI2", 
    model_name="gemini-2.5-flash-lite"
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Initializing RAGRetriever with model: gemini-2.5-flash-lite
Vector Store loaded with 2339 documents.
Building BM25 Index from Vector Store documents...
BM25 Index built with 2339 documents.
BM25 Index built with 2339 documents.


In [5]:
# Test 2: Syllabus
retriever.retrieve("Syllabus of Mathematics-I for first year")

--- 1. Hybrid Retrieval for: 'Syllabus of Mathematics-I for first year' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 19 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 4 | Title: Curricular Structure for B.Tech. I Year
  [1] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 0 | Title: Minimum Requirement to continue in the program & Promotion
  [2] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 4 | Title: Curricular Structure for B.Tech. I Year
  [3] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 5 | Title: Curricular Structure for B.Tech. I Year
  [4] Source: ../pdfs/Geotechnical_Engg.pdf | Page: 0 | Title: Ph.D. Entrance Exam Syllabus - Geotechnical Engineering
  [5] Source: ../pdfs/1st_Year_Scheme_Syallbus.pdf | Page: 0 | Title: Curricular Structure for B.Tech. I Year
  [6] Source: ../pdfs/Research_Areas_in_Space_2023.pdf | Page: 316 | Title: RESEARCH AREAS IN SPACE
  [7] So

[Document(id='6629a86c-ce04-4b99-86a9-8fc10173fb40', metadata={'doc_type': 'Syllabus', 'moddate': '2016-04-19T09:50:49+05:30', 'creationdate': '2016-04-19T09:50:49+05:30', 'year': 'Unknown', 'Dept': 'Institute', 'author': 'Valued Customer', 'creator': 'Microsoft® Word 2013', 'audience': 'UG', 'summary': 'This document outlines the curricular structure for the first year of the B.Tech. program, common to all branches at MNIT Jaipur.', 'producer': 'Microsoft® Word 2013', 'page': 4, 'title': 'Curricular Structure for B.Tech. I Year', 'source': '../pdfs/1st_Year_Scheme_Syallbus.pdf', 'total_pages': 15, 'page_label': '5'}, page_content='Differential Calculus :  Curvature , Concavity, convexity and points of  Inflexion, \nAsymptotes, Partial differentiation, Euler’s theorem on homogeneous functions, Total \ndifferentiation, Approximate calculation, Curve tracing (Cartesian and five polar curves - \nFolium of Descartes, Limacon, Cardioids, Lemniscates of Bernoulli and Equiangular \nspiral). \

In [6]:
# Test 3: Fee Structure (Specific Year)
retriever.retrieve("Fee Structure year 2016 admitted students for Btech students")

--- 1. Hybrid Retrieval for: 'Fee Structure year 2016 admitted students for Btech students' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 15 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/Final_Fee_Structure_PG_2017-18_admitted.pdf | Page: 1 | Title: Fee structure for M.Tech./M.Plan./MBA (Full-time) students
  [1] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [2] Source: ../pdfs/Fee_Structure_UG_2017-18.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [3] Source: ../pdfs/FeeUG2025-26.pdf | Page: 0 | Title: Fee Structure for B. Tech.
  [4] Source: ../pdfs/Fee_Structure_UG_2018-19_admitted.pdf | Page: 0 | Title: Fee Structure for B. Tech. /B.Arch.
  [5] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 2 | Title: Fee Structure for B. Tech. /B.Arch.
  [6] Source: ../pdfs/Fee_Structure_UG_2016-17.pdf | Page: 1 | Title: Fee Structure for B. Tech. /B.Arch.
  [7] Source:

[Document(id='5cec31ff-209e-4db2-8492-8cd5b1a2ff75', metadata={'moddate': '2017-04-27T18:21:24+05:30', 'doc_type': 'Fee Structure', 'title': 'Fee Structure for B. Tech. /B.Arch.', 'creationdate': '2017-04-27T18:21:24+05:30', 'author': 'IBM', 'creator': 'Microsoft® Word 2013', 'source': '../pdfs/Fee_Structure_UG_2016-17.pdf', 'total_pages': 3, 'summary': 'This document details the fee structure for B.Tech and B.Arch students admitted in the 2016-17 session.', 'year': '2016', 'page': 0, 'audience': 'UG', 'Dept': 'Institute', 'producer': 'Microsoft® Word 2013', 'page_label': '1'}, page_content='MALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \n Fee Structure for B. Tech. /B.Arch. students admitted in the session 2016-17 \n \nTUITION FEE \n \nS. No. \nHead of Fee \n \nOdd Semester & Even Semester \nOP/OBC PH/SC/ST \nIncome  \nBelow 1 Lac \nIncome                                  \n1 Lac to 5 Lac \nIncome                   \nAbove 5 Lac All \n1. Tuition Fee per Semester 0 20,834.00 6

In [ ]:
# Test 4: Faculty Query
retriever.retrieve("Quantum computing classes for faculty")

--- 1. Wide Retrieval for: 'Quantum computing classes for faculty' ---
--- 2. Reranking 10 documents ---
--- 2. Reranking 10 documents ---

--- Top Selected Documents ---
Score: 1.0 | Doc Source: ../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Quantum Sensing' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-03_BASIC_QUANTUM_PROGRAMMING_2025.pdf
Reasoning: This document is a notice for an 'Online Faculty Programme on Basic Quantum Programming' which is part of an 'AICTE Approved Minor Course Curriculum on Quantum Computing' for the year 2025, explicitly targeting faculty. This directly matches all user constraints.
Score: 1.0 | Doc Source: ../pdfs/QT-05_Quantum_Computation_Brochure_updated_1.pdf
Reasoning: This document is a notice for an 'Online Faculty P

[Document(id='168d713b-bef6-4d56-82f5-8998a0eda91a', metadata={'total_pages': 1, 'author': 'x', 'summary': 'An intensive 20-day online training programme on Quantum Sensing is being organized for faculty and doctoral students.', 'moddate': '2025-09-17T06:16:12+00:00', 'title': 'AICTE Approved Minor Course Curriculum on Quantum Computing', 'creator': 'Microsoft® Word 2016', 'doc_type': 'Notice', 'creationdate': '2025-09-17T06:16:12+00:00', 'page': 0, 'audience': 'Faculty', 'Dept': 'Institute', 'producer': 'www.ilovepdf.com', 'year': '2025', 'source': '../pdfs/QT-07_Quantum_Sensing_Brochure_upd.pdf', 'page_label': '1'}, page_content='AICTE Approved Minor Course Curriculum \non Quantum Computing \n                  \n  \n \nIIT Kanpur, IIT Roorkee, IIT Guwahati,     \nNIT Patna, NIT Warangal, IIITDM Jabalpur \nOnline Faculty Programme on  \nQT – 07 :  \nQuantum Sensing   \nSept 26 – Oct 17, 2025 \nTwenty Days (Mon to Sat) \nTime: 2 – 4 PM (Daily 2 Hours) \n \nAn intensive 20-day-40-hour T

In [7]:
# Test 5: Policy
retriever.retrieve("What is the unfair means policy policy or UFM of the instituion")

--- 1. Hybrid Retrieval for: 'What is the unfair means policy policy or UFM of the instituion' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 20 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 0 | Title: Minimum Requirement to continue in the program & Promotion
  [1] Source: ../pdfs/sou_motu_n.pdf | Page: 7 | Title: Organisation and Function
  [2] Source: ../pdfs/sou_motu_n.pdf | Page: 7 | Title: Organisation and Function
  [3] Source: ../pdfs/sou_motu_n.pdf | Page: 3 | Title: Organisation and Function
  [4] Source: ../pdfs/Semester_Promotion_Policy_2015.pdf | Page: 0 | Title: Promotion of B.Tech/B.Arch students
  [5] Source: ../pdfs/Research_Areas_in_Space_2023.pdf | Page: 277 | Title: RESEARCH AREAS IN SPACE
  [6] Source: ../pdfs/Notice_eligible_ineligible_applicants_for_the_post_Rregistrar.pdf | Page: 1 | Title: Final List of Eligible/ Ineligible Applicants for the Post of Registrar
  [7] S

[Document(id='eb6ab811-f286-4ee2-b0b7-53961f48adc2', metadata={'producer': 'Microsoft® Office Word 2007', 'moddate': '2015-04-23T17:12:10+05:30', 'page': 4, 'total_pages': 9, 'doc_type': 'Policy', 'audience': 'UG', 'creationdate': '2015-04-23T17:12:10+05:30', 'author': 'lavab', 'source': '../pdfs/Sem_promotion_policy_1st_year.pdf', 'year': 'Unknown', 'Dept': 'Institute', 'title': 'Minimum Requirement to continue in the program & Promotion', 'creator': 'Microsoft® Office Word 2007', 'summary': 'This document outlines the policy regarding minimum academic performance requirements for B.Tech./B.Arch. students to progress to subsequent semesters at MNIT Jaipur.', 'page_label': '5'}, page_content='Students who have failed in one semester / taken semester withdrawal / rusticated fo r one \nsemester / not promoted to higher semester on account of N -4 rule or any other reason \n/Academically deficient student  not able to register for higher semester courses due to \nregistration of pending  

In [8]:
# Test 5: Policy
retriever.retrieve("what are guidlines for pdf exam in year 2025")

--- 1. Hybrid Retrieval for: 'what are guidlines for pdf exam in year 2025' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 19 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/Guidelines.pdf | Page: 0 | Title: Ph.D. Entrance Exam Guidelines
  [1] Source: ../pdfs/06_Electrical_Interview_Schedule_Asst_Prof.pdf | Page: 2 | Title: Document Verification, Presentation and Personal Interview Schedule
  [2] Source: ../pdfs/Ranking_Document.pdf | Page: 48 | Title: Ranking Methodology for Engineering Institutions
  [3] Source: ../pdfs/01_AIDE_Interview_Schedule_Asst_Prof.pdf | Page: 1 | Title: Document Verification, Presentation and Personal Interview
  [4] Source: ../pdfs/sou_motu_n.pdf | Page: 4 | Title: Organisation and Function
  [5] Source: ../pdfs/06_Electrical_Interview_Schedule_Asst_Prof.pdf | Page: 1 | Title: Document Verification, Presentation and Personal Interview Schedule
  [6] Source: ../pdfs/Ranking_Methodology_Presentation.pdf | Pa

[Document(id='367df7d2-d7d6-44b5-a1ed-9b4710d0a514', metadata={'producer': 'Microsoft® Word 2013', 'author': 'wipro', 'source': '../pdfs/Guidelines.pdf', 'Dept': 'Institute', 'creationdate': '2025-12-02T14:48:48+05:30', 'moddate': '2025-12-02T14:48:48+05:30', 'creator': 'Microsoft® Word 2013', 'doc_type': 'Guidelines', 'title': 'Ph.D. Entrance Exam Guidelines', 'summary': 'This document outlines the guidelines for the Ph.D. entrance exam for the Even Semester 2025-26, including the selection process and requirements for shortlisted candidates.', 'page_label': '1', 'year': '2025-26', 'audience': 'UG', 'total_pages': 1, 'page': 0}, page_content='Guidelines for Ph.D. Entrance Exam, EVEN Semester 2025-26 \n \nWritten exam and interview for Ph.D. entrance exam, Even Semester 2025-26 will be \nheld during 09th and 10th December 2025 at MNIT campus.  \n \nSelection process will comprise of two steps (i) Written test (ii) Interview of \nshortlisted candidates. The written test will comprise of

In [10]:
# Test 5: Policy
retriever.retrieve("what is the college's perspective on Right to Information")

--- 1. Hybrid Retrieval for: 'what is the college's perspective on Right to Information' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 20 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/sou_motu_n.pdf | Page: 10 | Title: Organisation and Function
  [1] Source: ../pdfs/rti-act.pdf | Page: 3 | Title: THE RIGHT TO INFORMATION ACT, 2005
  [2] Source: ../pdfs/sou_motu_n.pdf | Page: 10 | Title: Organisation and Function
  [3] Source: ../pdfs/rti-act.pdf | Page: 1 | Title: THE RIGHT TO INFORMATION ACT, 2005
  [4] Source: ../pdfs/sou_motu_n.pdf | Page: 11 | Title: Organisation and Function
  [5] Source: ../pdfs/Grammarly User End Guide - Domain Users.pdf | Page: 1 | Title: How to Register
  [6] Source: ../pdfs/rti-act.pdf | Page: 0 | Title: THE RIGHT TO INFORMATION ACT, 2005
  [7] Source: ../pdfs/rti-act.pdf | Page: 13 | Title: THE RIGHT TO INFORMATION ACT, 2005
  [8] Source: ../pdfs/rti-act.pdf | Page: 6 | Title: THE RIGHT TO INFORMATION ACT

[Document(id='631bea8c-ef1f-4243-849b-f9434a54a0a8', metadata={'title': 'Organisation and Function', 'creationdate': '2024-09-30T17:27:55+05:30', 'page': 10, 'audience': 'Faculty', 'summary': 'Details regarding the organisation, functions, and duties of Malaviya National Institute of Technology, Jaipur are provided, referencing the NIT Act 2007 and Statutes.', 'producer': 'Microsoft® Word 2013', 'author': 'Administrator', 'creator': 'Microsoft® Word 2013', 'year': 'Unknown', 'source': '../pdfs/sou_motu_n.pdf', 'total_pages': 13, 'page_label': '11', 'doc_type': 'Notice', 'Dept': 'Institute', 'moddate': '2024-09-30T17:27:55+05:30'}, page_content='4.5.7 Frequently Asked Question (FAQs) http://mnit.ac.in/footer/rti.php \n4.5.8 Any other information such as - (a) Citizen’s Charter, (b) Result \nFramework Document (RFD), (c) Six monthly reports on the ,(d) Performance \nagainst the benchmarks set in the Citizen’s Charter \nhttp://mnit.ac.in/footer/rti.php \n4.6 Receipt & Disposal of RTI appl

In [12]:
retriever.retrieve("who were the gold medalist in year 2021-22")


--- 1. Hybrid Retrieval for: 'who were the gold medalist in year 2021-22' ---
  [Vector] Found 10 docs
  [Keyword] Found 10 docs
  [RRF] Fused into 20 unique docs

[Log] Top 10 Documents after Fusion:
  [0] Source: ../pdfs/Gold_Medalist.pdf | Page: 1 | Title: Director’s Gold Medal
  [1] Source: ../pdfs/rti-act.pdf | Page: 18 | Title: THE RIGHT TO INFORMATION ACT, 2005
  [2] Source: ../pdfs/Gold_Medalist.pdf | Page: 0 | Title: Director’s Gold Medal
  [3] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 6 | Title: Minimum Requirement to continue in the program & Promotion
  [4] Source: ../pdfs/18thScroll_2024.pdf | Page: 73 | Title: Scroll of Awardees
  [5] Source: ../pdfs/Sem_promotion_policy_1st_year.pdf | Page: 4 | Title: Minimum Requirement to continue in the program & Promotion
  [6] Source: ../pdfs/18thScroll_2024.pdf | Page: 74 | Title: Scroll of Awardees
  [7] Source: ../pdfs/Sem_promotion_policy_3sem.pdf | Page: 1 | Title: Minimum Requirement for B.Tech/B.Arch Continuat

[Document(id='b71cd549-0c1e-46b9-a3c5-7df7b3fa755a', metadata={'page_label': '1', 'creationdate': '2023-03-31T16:08:35+05:30', 'author': 'IBM', 'audience': 'UG', 'Dept': 'Institute', 'source': '../pdfs/Gold_Medalist.pdf', 'page': 0, 'summary': 'This document lists the recipients of the Director’s Gold Medal in B.Tech. and B.Arch. for the academic session 2021-22.', 'year': '2021-22', 'title': 'Director’s Gold Medal', 'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'doc_type': 'Notice', 'total_pages': 3, 'moddate': '2023-03-31T16:08:35+05:30'}, page_content='1 \n \n \n \n \nMALAVIYA NATIONAL INSTITUTE OF TECHNOLOGY JAIPUR \n \n \nDirector’s Gold Medal in B. Tech. and B. Arch. in the academic session \n2021-22 \n \nS. \nNo ID No. Name Programme CGPA \n1 2017UAR1567 NUPUR MALIK ARCHITECTURE AND \nPLANNING 9.41 \n2 2018UCH1656 DARSHANA \nPALIWAL CHEMICAL ENGINEERING 9.66 \n3 2018UCE1103 ARSHIKA TOMAR CIVIL ENGINEERING 9.70 \n4 2018UCP1444 PRANSHU VYAS COMPUTER SCIENC